# DX 704 Week 11 Project

In this project, you will develop and test prompts asking a language model to classify text from a home services query and match it to an appropriate category of home services.

The full project description and a template notebook are available on GitHub: [Project 11 Materials](https://github.com/bu-cds-dx704/dx704-project-11).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1 : Design a Short Prompt

The provided file "queries.txt" contains sample text from requests by homeowners by email or phone.
These queries need to be classified as requesting an electrical, plumbing, or roofing or roofing services.
The provided file has columns query_id, query, and target_category.
Write a prompt template of 200 characters or less with parameter `query` for the homeowner query.
Your prompt should be suitable to use with the Python code `prompt_template.format(query=query)`.
Test your prompt with the model `gemini-2.0-flash` and suitable parsing code.

In [2]:
import os
from google import genai

#load
api_key = os.getenv("GEMINI_API_KEY")
print("Key found?", bool(api_key))

client = genai.Client(api_key=api_key)

Key found? True


In [4]:
# YOUR CHANGES HERE
import pandas as pd

#set up prompt template (200 chars or less)
prompt_template = (
    "Classify this home service request as electrical, plumbing, or roofing. "
    "Reply with only one word: electrical, plumbing, or roofing.\n{query}"
)

#load query data
df = pd.read_csv("queries.txt", sep="\t")


#save prompt template to file
with open("short-prompt.txt", "w", encoding="utf-8") as f:
    f.write(prompt_template)

#parse gemini output
def parse_category(text: str) -> str:
    """
    Convert Gemini's response into one of:
    electrical, plumbing, roofing, unknown
    """
    text = text.strip().lower()

    if "electrical" in text:
        return "electrical"
    if "plumbing" in text:
        return "plumbing"
    if "roofing" in text:
        return "roofing"

    return "unknown"

In [5]:
#run predictions
predictions = []

for _, row in df.iterrows():
    query_id = row["query_id"]
    query = row["query"]
    
    prompt = prompt_template.format(query=query)
    
    try:
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt,
        )

        raw_output = response.text if hasattr(response, "text") and response.text else ""
        predicted_category = parse_category(raw_output)

    except Exception as e:
        raw_output = f"ERROR: {e}"
        predicted_category = "unknown"

    predictions.append({
        "query_id": query_id,
        "predicted_category": predicted_category,
        "raw_output": raw_output
    })

#save output file in df
results_df = pd.DataFrame(predictions)
results_df

,query_id,predicted_category,raw_output
0,1,roofing,roofing
1,2,plumbing,plumbing
2,3,electrical,electrical
3,4,roofing,roofing
4,5,plumbing,plumbing
5,6,electrical,electrical
6,7,roofing,roofing
7,8,plumbing,Plumbing
8,9,electrical,electrical
9,10,roofing,roofing


In [6]:
submission_df = results_df[["query_id", "predicted_category"]]
submission_df

,query_id,predicted_category
0,1,roofing
1,2,plumbing
2,3,electrical
3,4,roofing
4,5,plumbing
5,6,electrical
6,7,roofing
7,8,plumbing
8,9,electrical
9,10,roofing


Save your prompt template in a file "short-prompt.txt".
Save the results of your prompt testing in "short-output.tsv" with columns `query_id` and `predicted_category`.

In [ ]:
# YOUR CHANGES HERE

#save as csv
submission_df.to_csv("short-output.tsv", sep="\t", index=False)

Submit "short-prompt.txt" and "short-output.tsv" in Gradescope.

Hint: your prompt may be re-tested with the Gemini API, so do not rely solely on lucky language model responses.

## Part 2: Find Short Prompt Mistakes

Construct 5 queries of 100 characters or less that trick your short prompt so that the wrong category is chosen.


In [16]:
# YOUR CHANGES HERE
#construct 5 queries
mistake_candidates = pd.DataFrame([
    {"query": "Water leaking from ceiling below attic", "target_category": "plumbing"},
    {"query": "Need outlet installed in the bathroom for a toilet", "target_category": "electrical"},
    {"query": "Microwave trips breaker near sink leak", "target_category": "electrical"},
    {"query": "Lights flicker during bathroom floods", "target_category": "electrical"},
    {"query": "Ceiling light flickering in the bathroom shower", "target_category": "plumbing"},
    {"query": "Roof drip above electrical panel", "target_category": "electrical"},
    {"query": "Sink backing up after breaker tripped", "target_category": "plumbing"},
    {"query": "Water heater issue by electrical panel", "target_category": "plumbing"},
    {"query": "Water dripping from ceiling under attic", "target_category": "roofing"},
    {"query": "Need outlet installed for new bidet", "target_category": "plumbing"}
])

#classify queries
def classify_query(query):
    prompt = prompt_template.format(query=query)

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config={"temperature": 0}
    )
    raw_output = response.text if hasattr(response, "text") and response.text else ""
    predicted_category = parse_category(raw_output)

    return predicted_category, raw_output

mistakes = []

for _, row in mistake_candidates.iterrows():
    predicted_category, raw_output = classify_query(row["query"])
    mistakes.append({
        "query": row["query"],
        "target_category": row["target_category"],
        "predicted_category": predicted_category,
        "raw_output": raw_output,
        "is_mistake": predicted_category != row["target_category"]
    })

mistakes_df = pd.DataFrame(mistakes)
mistakes_df

,query,target_category,predicted_category,raw_output,is_mistake
0,Water leaking from ceiling below attic,plumbing,roofing,roofing,True
1,Need outlet installed in the bathroom for a to...,electrical,electrical,electrical,False
2,Microwave trips breaker near sink leak,electrical,electrical,electrical,False
3,Lights flicker during bathroom floods,electrical,electrical,electrical,False
4,Ceiling light flickering in the bathroom shower,plumbing,electrical,electrical,True
5,Roof drip above electrical panel,electrical,roofing,roofing,True
6,Sink backing up after breaker tripped,plumbing,plumbing,Plumbing,False
7,Water heater issue by electrical panel,plumbing,electrical,electrical,True
8,Water dripping from ceiling under attic,roofing,plumbing,Plumbing,True
9,Need outlet installed for new bidet,plumbing,electrical,electrical,True


In [18]:
true_mistakes_df = mistakes_df[mistakes_df["is_mistake"]].copy()
true_mistakes_df

,query,target_category,predicted_category,raw_output,is_mistake
0,Water leaking from ceiling below attic,plumbing,roofing,roofing,True
4,Ceiling light flickering in the bathroom shower,plumbing,electrical,electrical,True
5,Roof drip above electrical panel,electrical,roofing,roofing,True
7,Water heater issue by electrical panel,plumbing,electrical,electrical,True
8,Water dripping from ceiling under attic,roofing,plumbing,Plumbing,True
9,Need outlet installed for new bidet,plumbing,electrical,electrical,True


In [19]:
final_mistakes_df = true_mistakes_df[["query", "target_category", "predicted_category"]].head(5)
final_mistakes_df

,query,target_category,predicted_category
0,Water leaking from ceiling below attic,plumbing,roofing
4,Ceiling light flickering in the bathroom shower,plumbing,electrical
5,Roof drip above electrical panel,electrical,roofing
7,Water heater issue by electrical panel,plumbing,electrical
8,Water dripping from ceiling under attic,roofing,plumbing


Save your 5 queries in a file "mistakes.tsv" with columns `query`, `target_category` and `predicted_category`.

In [20]:
# YOUR CHANGES HERE
final_mistakes_df.to_csv("mistakes.tsv", sep="\t", index=False)

Submit "mistakes.tsv" in Gradescope.

## Part 3: Design a Long Prompt

Repeat part 1 with a length limit of 5000 characters.

In [21]:
# YOUR CHANGES HERE

prompt_template = """
You are classifying a homeowner's service request into exactly one category.

The following categories are allowed:
- electrical
- plumbing
- roofing

Category Definitions:
- electrical: anything involving wiring, outlets, switches, breakers, circuits, lights, ceiling fans, panels, garage door openers, Ethernet wiring, or electrical installation/repair.
- plumbing: anything involving sinks, toilets, showers, faucets, drains, pipes, sewage smells, garbage disposals, boilers, bidets, water heaters, flooding, water pressure, or plumbing installation/repair.
- roofing: anything involving roofs, shingles, leaks from the roof, attic roof issues, storm or hurricane roof damage, flashing, patching, inspections of roof damage, roof replacement, tile roofs, metal roofs, or insurance work for roof damage.

Rules:
1. Return exactly one label: electrical, plumbing, or roofing.
2. Do not explain your answer.
3. If a request mentions multiple issues, choose the category that best matches the main service being requested.
4. If the request involves water coming from a roof, attic, shingles, storm damage, or a tree hitting the roof, choose roofing.
5. If the request involves fixtures or appliances that use water or drain water, choose plumbing.
6. If the request involves power, wiring, breakers, outlets, switches, lighting, or electrical devices, choose electrical.

Homeowner request:
{query}
"""

Save your longer prompt template in a file "long-prompt.txt".
Save the results of your prompt testing in "long-output.tsv".
Both files should use the same columns as part 1.

In [22]:
# YOUR CHANGES HERE
#save prompt template
with open("long-prompt.txt", "w", encoding="utf-8") as f:
    f.write(prompt_template)

#parse Gemini output
def parse_category(text: str) -> str:
    text = text.strip().lower()

    if "electrical" in text:
        return "electrical"
    if "plumbing" in text:
        return "plumbing"
    if "roofing" in text:
        return "roofing"

    return "unknown"

#run predictions
predictions = []

for _, row in df.iterrows():
    query_id = row["query_id"]
    query = row["query"]

    prompt = prompt_template.format(query=query)

    try:
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt,
            config={"temperature": 0}
        )

        raw_output = response.text if hasattr(response, "text") and response.text else ""
        predicted_category = parse_category(raw_output)

    except Exception as e:
        raw_output = f"ERROR: {e}"
        predicted_category = "unknown"

    predictions.append({
        "query_id": query_id,
        "predicted_category": predicted_category,
        "raw_output": raw_output
    })

#save in df
results_df = pd.DataFrame(predictions)
submission_df = results_df[["query_id", "predicted_category"]]
print(results_df.head(5))
print(submission_df.head(5))

   query_id predicted_category  raw_output
0         1            roofing     roofing
1         2           plumbing    plumbing
2         3         electrical  electrical
3         4            roofing     roofing
4         5           plumbing    plumbing
   query_id predicted_category
0         1            roofing
1         2           plumbing
2         3         electrical
3         4            roofing
4         5           plumbing


In [23]:
submission_df.to_csv("long-output.tsv", sep="\t", index=False)

Submit "long-prompt.txt" and "long-output.tsv" in Gradescope.

## Part 4: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 5: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.

In [24]:
with open('acknowledgments.txt', 'w') as f:
    f.write("When working on this project, I referenced the following websites:")
    f.write("https://ai.google.dev/gemini-api/docs")
    f.write("https://ai.google.dev/gemini-api/docs/prompting-strategies")